# CNN Model for Patient Age Regression

This notebook implements and verifies the CNN model for patient age regression from NIH ChestX-ray14.

### Model Pipeline

**Input X-ray → CNN → Global Average Pooling → Linear(1) → Predicted Age**

### Model Contract

- Input shape: `[B, 1, 224, 224]`
- Output shape: `[B, 1]`
- Task: Regression
- Target: Patient Age


In [7]:
import sys
from pathlib import Path

import torch

MODULE_PATH = Path.cwd().parent
sys.path.append(str(MODULE_PATH))

from src.model import AgeRegressionCNN

In [8]:
model = AgeRegressionCNN()

print(model)

AgeRegressionCNN(
  (features): Sequential(
    (0): ConvBlock(
      (block): Sequential(
        (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (1): ConvBlock(
      (block): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (2): ConvBlock(
      (block): Sequential(
        (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=Tru

## 2. Tensor Shape Verification

The model receives grayscale chest X-ray images with shape `[B, 1, 224, 224]`.

The tensor shape is checked after each convolutional block, Global Average Pooling, flattening, and the final regression layer.

In [9]:
x = torch.randn(4, 1, 224, 224)

print("Input:", x.shape)

with torch.no_grad():
    for i, block in enumerate(model.features, start=1):
        x = block(x)
        print(f"After Conv Block {i}:", x.shape)

    x = model.global_avg_pool(x)
    print("After GAP:", x.shape)

    x = torch.flatten(x, 1)
    print("After Flatten:", x.shape)

    x = model.regressor(x)
    print("After Linear:", x.shape)

Input: torch.Size([4, 1, 224, 224])
After Conv Block 1: torch.Size([4, 32, 112, 112])
After Conv Block 2: torch.Size([4, 64, 56, 56])
After Conv Block 3: torch.Size([4, 128, 28, 28])
After Conv Block 4: torch.Size([4, 256, 14, 14])
After GAP: torch.Size([4, 256, 1, 1])
After Flatten: torch.Size([4, 256])
After Linear: torch.Size([4, 1])


## 3. Trainable Parameter Count

The number of trainable parameters is calculated to measure the size of the CNN model.

In [10]:
total_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Trainable parameters: {total_params:,}")

Trainable parameters: 389,057


## 4. Forward Pass Test

A forward pass is performed using a sample batch of grayscale X-ray images to verify that the CNN produces one age prediction for each input image.


In [11]:
model.eval()

sample_input = torch.randn(4, 1, 224, 224)

with torch.no_grad():
    predictions = model(sample_input)

print("Input shape:", sample_input.shape)
print("Prediction shape:", predictions.shape)
print("Predicted ages:", predictions.squeeze(1))

Input shape: torch.Size([4, 1, 224, 224])
Prediction shape: torch.Size([4, 1])
Predicted ages: tensor([0.0166, 0.0161, 0.0161, 0.0153])


## 5. Architecture Explanation

The CNN model is designed to extract visual features from grayscale chest X-ray images and regress them to a continuous patient age.

### Convolutional Blocks

Each convolutional block consists of:

- `Conv2d`: extracts local visual features from the X-ray image.
- `BatchNorm2d`: stabilizes the feature distributions during training.
- `ReLU`: introduces non-linearity into the network.
- `MaxPool2d`: reduces the spatial resolution while retaining important features.

The four convolutional blocks progressively increase the number of feature channels:

`1 → 32 → 64 → 128 → 256`

while reducing the spatial resolution:

`224 → 112 → 56 → 28 → 14`

### Global Average Pooling

Global Average Pooling converts the final feature maps from `[B, 256, 14, 14]` to `[B, 256, 1, 1]`.

This produces one representative value for each feature channel and reduces the number of features passed to the regression layer.

### Regression Layer

The flattened 256-dimensional feature vector is passed to a linear layer:

`Linear(256 → 1)`

The output is one continuous value representing the predicted patient age.

## 6. Model Architecture Summary

| Layer | Output Shape | Purpose |
|---|---|---|
| Input | `[B, 1, 224, 224]` | Grayscale chest X-ray |
| Conv Block 1 | `[B, 32, 112, 112]` | Low-level feature extraction |
| Conv Block 2 | `[B, 64, 56, 56]` | Higher-level feature extraction |
| Conv Block 3 | `[B, 128, 28, 28]` | Deeper visual representation |
| Conv Block 4 | `[B, 256, 14, 14]` | High-level feature extraction |
| Global Average Pooling | `[B, 256, 1, 1]` | Spatial feature aggregation |
| Flatten | `[B, 256]` | Convert features to a vector |
| Linear | `[B, 1]` | Predict patient age |

**Trainable parameters:** 389,057